[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-05-autologging.ipynb#scrollTo=10a2b3c4)

---
# Day 5 · Autologging with sklearn, XGBoost, and PyTorch
**certified-journeys / mlflow-certified** · Learn session

> **Goal for today:** Enable global autologging with `mlflow.autolog()`, train models with sklearn and XGBoost, understand what gets captured automatically, and compare autologged output against manual logging for the same model.


In [ ]:
%pip install -q mlflow scikit-learn xgboost pandas numpy


## Step 1 · What is autologging and how does it work?

MLflow **autologging** intercepts framework training calls and automatically logs:
- Hyperparameters (from the estimator's `get_params()`)
- Training metrics (accuracy, loss, eval metrics per round)
- The trained model artifact
- Feature importances, confusion matrix (sklearn)

| Call | Scope |
|---|---|
| `mlflow.autolog()` | All supported frameworks globally |
| `mlflow.sklearn.autolog()` | Only sklearn |
| `mlflow.xgboost.autolog()` | Only XGBoost |
| `mlflow.pytorch.autolog()` | Only PyTorch |

Autologging is **opt-in** — you enable it once before training and MLflow handles the rest.


In [ ]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import xgboost as xgb

# Local tracking URI — works in Colab without a server
mlflow.set_tracking_uri("./mlruns")

# Load data once — reuse across all experiments in this notebook
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Dataset: breast cancer | train={X_train.shape[0]}, test={X_test.shape[0]}")
print(f"MLflow version: {mlflow.__version__}")


**What just happened?**

- Importing `mlflow.sklearn` and `mlflow.xgboost` loads the autolog integration modules — they patch the framework's fit methods when autologging is enabled.
- We reuse the breast cancer dataset so results across all experiments in this notebook are directly comparable.
- **No autologging is active yet** — `mlflow.autolog()` must be called before training for it to take effect.


## Step 2 · Enable global autologging and train an sklearn Pipeline

A `Pipeline` chains preprocessing and model training into a single estimator.  
MLflow's sklearn autologging handles Pipelines natively — it logs the final estimator's params and wraps the whole pipeline as the saved model artifact.

**Key autolog options:**

| Parameter | Default | Effect |
|---|---|---|
| `log_input_examples` | False | Logs a sample of training data |
| `log_model_signatures` | True | Infers and logs input/output schema |
| `log_models` | True | Saves the trained model as an artifact |
| `max_tuning_runs` | 5 | Limits child runs from GridSearchCV |


In [ ]:
# Enable global autologging — covers all supported frameworks
# Call this BEFORE any model training
mlflow.autolog(
    log_input_examples=True,
    log_model_signatures=True,
    log_models=True,
    silent=False,  # set True in production to suppress autolog warnings
)

mlflow.set_experiment("day05-autologging-sklearn")

with mlflow.start_run(run_name="sklearn-pipeline-autolog") as run:
    # Build a Pipeline: StandardScaler -> LogisticRegression
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=0.5,
            max_iter=300,
            solver="lbfgs",
            random_state=42,
        )),
    ])

    # Calling .fit() triggers autologging — no manual log_param/log_metric needed
    pipeline.fit(X_train, y_train)

    # We can still add manual metrics on top of autologged ones
    y_pred = pipeline.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)  # manual addition

    sklearn_run_id = run.info.run_id

print(f"Run ID: {sklearn_run_id[:8]}...")
print(f"Test accuracy (manually logged): {test_acc:.4f}")


**What just happened?**

- `mlflow.autolog()` patched `Pipeline.fit()` — when we called `pipeline.fit()`, MLflow automatically logged all `LogisticRegression` hyperparameters (`C`, `max_iter`, `solver`, etc.).
- The model was saved as an artifact under `model/` in the run directory, ready to be loaded with `mlflow.sklearn.load_model()`.
- **We added `test_accuracy` manually** — autologging logs training metrics (cross-val score) but you need to manually log held-out test set metrics.
- `silent=False` lets you see which params/metrics autologging captured in the output.


## Step 3 · Inspect the autologged artifacts

After training, let's use `MlflowClient` to inspect exactly what autologging captured — params, metrics, and artifacts.  
This is the programmatic equivalent of opening the MLflow UI run detail page.


In [ ]:
from mlflow.tracking import MlflowClient
import os

client = MlflowClient()
run_data = client.get_run(sklearn_run_id)

print("=== Autologged PARAMS ===")
for k, v in sorted(run_data.data.params.items()):
    print(f"  {k}: {v}")

print("\n=== Autologged METRICS ===")
for k, v in sorted(run_data.data.metrics.items()):
    print(f"  {k}: {v:.4f}")

print("\n=== Autologged ARTIFACTS ===")
artifacts = client.list_artifacts(sklearn_run_id)
for art in artifacts:
    print(f"  {art.path}  ({'dir' if art.is_dir else f'{art.file_size} bytes'})")


**What just happened?**

- The params list shows everything from `Pipeline.get_params()` — note the `classifier__` prefix (sklearn's step-naming convention).
- Metrics include `training_accuracy_score` (cross-val) and our manually logged `test_accuracy`.
- **Artifacts include the `model/` directory** — contains `MLmodel`, `model.pkl`, `requirements.txt`, and `conda.yaml`.
- The `MLmodel` file is the key — it describes model flavors, signature, and how to load the model.


## Step 4 · XGBoost autologging — eval metrics per round

XGBoost's autologging captures something sklearn can't: **per-round eval metrics**.  
When you pass an `evals` list to `xgb.train()`, MLflow logs the metric at every boosting round — enabling loss curve analysis.

This is especially useful for:
- Detecting overfitting (train loss drops while val loss rises)
- Choosing the optimal `num_round` without re-running the experiment


In [ ]:
import mlflow.xgboost

# autolog() is already active globally; XGBoost is covered automatically
mlflow.set_experiment("day05-autologging-xgboost")

# XGBoost native API uses DMatrix objects
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test,  label=y_test)

params = {
    "objective":  "binary:logistic",
    "eval_metric": "logloss",
    "max_depth":   4,
    "eta":         0.1,       # learning rate
    "subsample":   0.8,
    "seed":        42,
}

with mlflow.start_run(run_name="xgb-native-autolog") as run:
    # evals list tells XGBoost to evaluate on both train and test at each round
    # MLflow autologging intercepts this and logs metrics per step
    evals_result = {}
    booster = xgb.train(
        params,
        dtrain,
        num_boost_round=50,
        evals=[(dtrain, "train"), (dtest, "test")],
        evals_result=evals_result,
        verbose_eval=10,  # print every 10 rounds
    )

    # Manual: log final test accuracy
    y_pred_proba = booster.predict(dtest)
    y_pred = (y_pred_proba > 0.5).astype(int)
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)

    xgb_run_id = run.info.run_id

print(f"\nXGBoost run ID: {xgb_run_id[:8]}...")
print(f"Final test accuracy: {test_acc:.4f}")
print(f"Final train logloss: {evals_result['train']['logloss'][-1]:.4f}")
print(f"Final test  logloss: {evals_result['test']['logloss'][-1]:.4f}")


**What just happened?**

- XGBoost autologging logged `train-logloss` and `test-logloss` at **every one of the 50 rounds** — visible as a time-series chart in the MLflow UI.
- The `evals_result` dict is local to the script; MLflow autologging captures the same data independently.
- **`params` dict was logged automatically** — `max_depth`, `eta`, `subsample`, `objective` all appear in the run.
- The trained `booster` was saved as an XGBoost artifact (`.json` model file) — can be loaded with `mlflow.xgboost.load_model()`.


## Step 5 · Selectively disable autologging for specific libraries

Sometimes you want autologging for some frameworks but not others — e.g., you have a custom sklearn wrapper where autologging captures the wrong params.

**Options:**
- `mlflow.sklearn.autolog(disable=True)` — disables only sklearn, leaves xgboost active
- `mlflow.autolog(disable=True)` — disables all autologging globally
- Per-framework call: call each framework's `.autolog()` individually instead of the global `mlflow.autolog()`


In [ ]:
# Disable sklearn autologging while keeping XGBoost active
mlflow.sklearn.autolog(disable=True)

mlflow.set_experiment("day05-manual-vs-autolog")

with mlflow.start_run(run_name="manual-logging-sklearn") as run:
    # With sklearn autologging disabled, we log everything manually
    model = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Must log everything ourselves now
    mlflow.log_param("n_estimators",  model.n_estimators)
    mlflow.log_param("learning_rate", model.learning_rate)
    mlflow.log_param("max_depth",     model.max_depth)
    mlflow.log_metric("test_accuracy", acc)
    mlflow.set_tag("logging_mode", "manual")

    manual_run_id = run.info.run_id

print(f"Manual run — test accuracy: {acc:.4f}")

# Re-enable sklearn autologging for the comparison run
mlflow.sklearn.autolog(disable=False, log_models=True)

with mlflow.start_run(run_name="autolog-sklearn") as run:
    # Exact same model config — autologging captures params/metrics automatically
    model2 = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
    )
    model2.fit(X_train, y_train)
    y_pred2 = model2.predict(X_test)
    acc2 = accuracy_score(y_test, y_pred2)
    mlflow.log_metric("test_accuracy", acc2)  # still log test_acc manually
    mlflow.set_tag("logging_mode", "autolog")

    autolog_run_id = run.info.run_id

print(f"Autolog run — test accuracy: {acc2:.4f}")


**What just happened?**

- `mlflow.sklearn.autolog(disable=True)` turns off sklearn patching — `.fit()` runs normally with no autologging side effects.
- `mlflow.sklearn.autolog(disable=False)` re-enables it — the patch is reapplied for the next `.fit()` call.
- **The autolog run will have more params** — autologging captures internal estimator state (e.g. `validation_fraction`, `n_iter_no_change`) that manual logging often misses.
- Tip: **always keep autologging on unless you have a specific reason** — you can always ignore extra params.


## Step 6 · Compare manual vs autolog output side-by-side

Let's use `mlflow.search_runs()` to pull both runs and compare what each logging approach captured.


In [ ]:
exp = mlflow.get_experiment_by_name("day05-manual-vs-autolog")

df = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["start_time ASC"],
)

# Separate param and metric columns
param_cols   = [c for c in df.columns if c.startswith("params.")]
metric_cols  = [c for c in df.columns if c.startswith("metrics.")]

print(f"Manual run  — params logged: {df[df['tags.logging_mode']=='manual'][param_cols].notna().sum(axis=1).iloc[0]}")
print(f"Autolog run — params logged: {df[df['tags.logging_mode']=='autolog'][param_cols].notna().sum(axis=1).iloc[0]}")
print(f"\nParams only in autolog run:")

manual_row  = df[df["tags.logging_mode"] == "manual"].iloc[0]
autolog_row = df[df["tags.logging_mode"] == "autolog"].iloc[0]

for col in param_cols:
    manual_val  = manual_row[col]
    autolog_val = autolog_row[col]
    if pd.isna(manual_val) and pd.notna(autolog_val):
        print(f"  {col.replace('params.','')} = {autolog_val}")

print(f"\nMetric comparison:")
for col in metric_cols:
    m = manual_row[col]  if pd.notna(manual_row[col])  else 'not logged'
    a = autolog_row[col] if pd.notna(autolog_row[col]) else 'not logged'
    print(f"  {col.replace('metrics.',''): <35} manual={m!s:>8}  autolog={a!s:>8}")


**What just happened?**

- The autolog run captures significantly more params — internal defaults like `validation_fraction` and `tol` that manual logging skips entirely.
- **Metrics are comparable** — both runs log `test_accuracy`, but autologging also adds `training_accuracy_score` from cross-validation.
- The difference in param count illustrates why autologging is preferred: **you can't accidentally forget to log a param that affects reproducibility**.
- Manual logging is useful when you need custom metric names or want to log domain-specific derived metrics.


In [ ]:
# Challenge: Extend autologging to a new scenario
#
# 1. Enable autologging globally, then train a Pipeline that chains:
#    - PCA(n_components=10)
#    - RandomForestClassifier(n_estimators=100, max_depth=5)
#    Verify that params for BOTH steps appear in the run.
#
# 2. Train an XGBoost model with num_boost_round=100 and log
#    a custom metric 'test_auc' using roc_auc_score.
#    (mlflow.log_metric can be called inside the same run as autologging)
#
# 3. Disable XGBoost autologging selectively with mlflow.xgboost.autolog(disable=True)
#    and verify that a subsequent xgb.train() call logs NO params automatically.
#    Hint: check run_data.data.params is empty after training

# Your solution here:
# from sklearn.decomposition import PCA
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import roc_auc_score
# mlflow.autolog()
# ...


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `mlflow.autolog()` | One call before training captures params, metrics, and model for all supported frameworks |
| `mlflow.sklearn.autolog(disable=True)` | Disables only sklearn; other frameworks stay active |
| XGBoost per-round metrics | Autologging logs `evals` metrics at every boosting round — enables loss curve analysis |
| Manual + autolog coexist | `mlflow.log_metric()` can be called inside the same run — adds custom metrics on top |
| Pipeline autologging | Params are prefixed with step name (`classifier__C`) from `Pipeline.get_params()` |

> **Tip:** `mlflow.autolog()` is a one-liner that covers most workflows — only add manual `log_metric` calls when you need custom metrics autologging doesn't capture.

---
## What's next
**Day 6** → MLflow Models and Flavors — learn how to log, inspect, and reload models using sklearn, pyfunc, and ONNX flavors; add model signatures and input examples.

Mark Day 5 complete in your [tracker](../index.html).
